# Session 8b — Regression-Based Event Studies: AR Models, GARCH, and Event Dummies

**Course: Event Studies in Finance & Economics**

*Mathis Mourey*

---

The standard event study methodology developed in Sessions 1 through 8 follows a two-step logic: estimate normal returns in the estimation window, then measure abnormal returns in the event window. This approach treats the estimation and event windows as separate regimes and processes each event independently. When the researcher studies a single entity (a firm, an index, a sovereign bond) exposed to *multiple* events over time, this separation becomes wasteful. The events are embedded in a single return series, and a regression model fitted to the full series can estimate all event effects simultaneously while controlling for the time-series dynamics of the return process.

This session develops the regression-based event study, where event dummies are inserted directly into a time-series model of returns. We begin with the autoregressive (AR) model augmented with event dummies, then extend to the GARCH family to model time-varying volatility. The GARCH framework is particularly valuable because it jointly estimates the event's impact on the conditional mean (the abnormal return) and on the conditional variance (the event-induced volatility change), addressing the problem that motivated the BMP test in Session 4.

The application throughout this session is the impact of OPEC production announcements on crude oil futures returns. OPEC is a single entity that generates multiple events (production decisions at scheduled meetings, emergency announcements, informal signals) over many years. The events are well-dated, economically important, and generate both mean and variance effects, making this an ideal setting for the regression-based approach.

**References for this session:**

- Bera, A.K. and Higgins, M.L. (1993). ARCH Models: Properties, Estimation and Testing. *Journal of Economic Surveys*, 7(4), 305--366.
- Bollerslev, T. (1986). Generalized Autoregressive Conditional Heteroskedasticity. *Journal of Econometrics*, 31(3), 307--327.
- Engle, R.F. (1982). Autoregressive Conditional Heteroscedasticity with Estimates of the Variance of United Kingdom Inflation. *Econometrica*, 50(4), 987--1007.
- Karafiath, I. (1988). Using Dummy Variables in the Event Methodology. *Financial Review*, 23(3), 351--357.
- Lunde, A. and Timmermann, A. (2004). Duration Dependence in Stock Prices: An Analysis of Bull and Bear Markets. *Journal of Business and Economic Statistics*, 22(3), 253--273.
- Salinger, M. (1992). Standard Errors in Event Studies. *Journal of Financial and Quantitative Analysis*, 27(1), 39--53.
- Savickas, R. (2003). Event-Induced Volatility and Tests for Abnormal Performance. *Journal of Financial Research*, 26(2), 165--178.

## 1. From Two-Step to Single-Equation Event Studies

### 1.1 The Standard Approach and Its Limitations

The standard event study estimates the market model over the estimation window, computes abnormal returns in the event window, and aggregates across firms or events. When applied to a single entity with multiple events, this approach has three limitations.

First, each event is processed independently. The estimation window for event $k$ excludes the event windows of all other events, which wastes data and creates gaps in the time series. If the events are frequent (e.g., monthly OPEC meetings), the estimation windows shrink and the parameter estimates become imprecise.

Second, the market model is assumed to be stable across estimation and event windows. If the entity's beta or volatility changes over time (as is likely for oil-sensitive assets), this assumption introduces bias.

Third, the standard approach estimates the abnormal return as a residual and then tests it in a separate step. This two-step procedure does not allow for joint estimation of the event effect and the return dynamics, and it ignores the uncertainty in the first-step parameter estimates when constructing the second-step test statistics (the generated regressor problem discussed in Session 6).

### 1.2 The Regression-Based Alternative

The regression-based event study embeds the event directly in the return model. The simplest specification augments the market model with an event dummy:

$$
R_{i,t} = \alpha + \beta R_{m,t} + \sum_{k=1}^{K} \gamma_k D_{k,t} + \varepsilon_t \tag{1}
$$

where $D_{k,t}$ is a dummy variable equal to one during the event window of event $k$ and zero otherwise, and $\gamma_k$ is the abnormal return associated with event $k$. This model is estimated by OLS over the *entire* sample period (estimation and event windows combined). The coefficient $\gamma_k$ is algebraically identical to the abnormal return from the two-step procedure when the event windows do not overlap and the estimation window covers the full non-event period (Karafiath, 1988; Salinger, 1992).

The advantage of the regression approach is that it estimates all event effects simultaneously, uses all available data, and produces standard errors from a single regression that account for the estimation uncertainty in $\alpha$ and $\beta$. It also extends naturally to richer time-series specifications, as we develop below.

### 1.3 Multi-Day Event Windows

When the event window spans multiple days (e.g., $[-1, +1]$), the dummy variable is one for all days in the window. The coefficient $\gamma_k$ then measures the average daily abnormal return during the window. To obtain the CAR, multiply by the number of days in the window: $CAR_k = \gamma_k \times T_w$.

Alternatively, one can use separate dummies for each day in the event window:

$$
R_{i,t} = \alpha + \beta R_{m,t} + \gamma_{k,pre} D_{k,t}^{pre} + \gamma_{k,0} D_{k,t}^{0} + \gamma_{k,post} D_{k,t}^{post} + \varepsilon_t \tag{2}
$$

This specification allows the pre-event, event-day, and post-event abnormal returns to differ, which is useful for studying the dynamics of the price adjustment (anticipation, immediate reaction, post-event drift).

## 2. Application: OPEC Production Announcements and Oil Returns

### 2.1 Why OPEC?

OPEC (the Organization of the Petroleum Exporting Countries) provides an ideal setting for the regression-based event study. OPEC holds scheduled ministerial conferences approximately every six months (more frequently during crises), at which members agree on production targets. These decisions are announced at known dates and have large, well-documented effects on oil prices. Because the same entity (OPEC) generates multiple events over a long time series, the single-entity, multiple-event framework applies directly.

The key economic mechanism is straightforward. An announcement of production cuts reduces expected future supply, raising oil prices. An announcement of production increases (or failure to agree on cuts) lowers prices. The magnitude depends on the size of the cut relative to expectations, the credibility of compliance, and the broader macroeconomic context.

### 2.2 Sample

We use daily returns on the USO ETF (United States Oil Fund) as a proxy for crude oil futures returns, with the S&P 500 as the market factor. The sample covers 2019 through 2023, a period that includes the historic OPEC+ production cut of April 2020 (in response to the COVID demand collapse), the gradual unwinding of cuts in 2021-2022, and the production adjustments around the Russia-Ukraine war. We select a set of major OPEC/OPEC+ announcements with clear production signals.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import statsmodels.api as sm
from arch import arch_model
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# -- OPEC announcement dates and direction of production decision --
opec_events = pd.DataFrame({
    'date': pd.to_datetime([
        '2020-03-06',  # OPEC-Russia talks collapse → no cut → prices crash
        '2020-04-12',  # Historic 9.7 mb/d cut agreed
        '2020-06-06',  # Cut extended by 1 month
        '2020-12-03',  # Gradual increase of 0.5 mb/d from Jan
        '2021-04-01',  # Gradual increase May-Jul
        '2021-07-18',  # Increase 0.4 mb/d monthly from Aug
        '2022-03-02',  # Maintain modest 0.4 mb/d increases
        '2022-10-05',  # Surprise 2 mb/d cut
        '2023-04-02',  # Surprise voluntary cuts ~1.16 mb/d
        '2023-06-04',  # Saudi Arabia 1 mb/d voluntary cut from July
        '2023-11-30',  # Voluntary cuts extended + deepened
    ]),
    'decision': [
        'No agreement (bearish)',
        'Historic cut 9.7mb/d (bullish)',
        'Cut extended (bullish)',
        'Gradual increase (bearish)',
        'Gradual increase (bearish)',
        'Increase 0.4mb/d monthly (bearish)',
        'Modest increase (neutral)',
        'Surprise 2mb/d cut (bullish)',
        'Surprise voluntary cuts (bullish)',
        'Saudi 1mb/d voluntary cut (bullish)',
        'Voluntary cuts extended (bullish)',
    ],
    'direction': [-1, 1, 1, -1, -1, -1, 0, 1, 1, 1, 1],
})

# Download data
oil_ticker = 'USO'
market_ticker = '^GSPC'
start_date = '2019-01-01'
end_date = '2024-01-01'

oil_data = yf.download(oil_ticker, start=start_date, end=end_date, progress=False)
mkt_data = yf.download(market_ticker, start=start_date, end=end_date, progress=False)

for d in [oil_data, mkt_data]:
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)

oil_ret = oil_data['Close'].pct_change().dropna() * 100  # in percent
mkt_ret = mkt_data['Close'].pct_change().dropna() * 100

common = oil_ret.index.intersection(mkt_ret.index)
oil_ret = oil_ret.loc[common]
mkt_ret = mkt_ret.loc[common]

print(f"Sample: {len(common)} trading days ({common[0].date()} to {common[-1].date()})")
print(f"\nOPEC Events: {len(opec_events)} announcements")
print(opec_events[['date', 'decision']].to_string(index=False))

## 3. The AR Model with Event Dummies

We first estimate a simple specification: an AR(1) model of oil returns augmented with the market return and event dummies. The AR(1) term captures any short-term serial correlation in oil returns (which is small but may be present due to microstructure effects). The event dummies cover a $[-1, +1]$ window around each announcement.

The model is:

$$
R_{oil,t} = \alpha + \phi R_{oil,t-1} + \beta R_{m,t} + \sum_{k=1}^{K} \gamma_k D_{k,t} + \varepsilon_t \tag{3}
$$

where $\phi$ is the AR(1) coefficient and $\gamma_k$ is the average daily abnormal return during the event window of event $k$. The CAR for event $k$ over the 3-day window is $3 \times \gamma_k$.

We also estimate a pooled specification that distinguishes bullish (production cut) and bearish (production increase or no agreement) events:

$$
R_{oil,t} = \alpha + \phi R_{oil,t-1} + \beta R_{m,t} + \gamma^{+} D_t^{bullish} + \gamma^{-} D_t^{bearish} + \varepsilon_t \tag{4}
$$

This tests whether the average effect of cuts and increases is symmetric.

In [ ]:
# -- Construct event dummies --
trading_days = oil_ret.index
df = pd.DataFrame({'oil': oil_ret, 'market': mkt_ret}, index=trading_days)
df['oil_lag'] = df['oil'].shift(1)

# Individual event dummies ([-1, +1] window)
for i, row in opec_events.iterrows():
    edate = row['date']
    eidx = trading_days.get_indexer([edate], method='ffill')[0]
    col_name = f'D_{i}'
    df[col_name] = 0
    for offset in [-1, 0, 1]:
        idx = eidx + offset
        if 0 <= idx < len(trading_days):
            df.iloc[idx, df.columns.get_loc(col_name)] = 1

# Pooled dummies: bullish vs bearish
df['D_bullish'] = 0
df['D_bearish'] = 0
for i, row in opec_events.iterrows():
    edate = row['date']
    eidx = trading_days.get_indexer([edate], method='ffill')[0]
    for offset in [-1, 0, 1]:
        idx = eidx + offset
        if 0 <= idx < len(trading_days):
            if row['direction'] > 0:
                df.iloc[idx, df.columns.get_loc('D_bullish')] = 1
            elif row['direction'] < 0:
                df.iloc[idx, df.columns.get_loc('D_bearish')] = 1

df = df.dropna()

# Model A: Individual event dummies
dummy_cols = [f'D_{i}' for i in range(len(opec_events))]
X_indiv = sm.add_constant(df[['oil_lag', 'market'] + dummy_cols])
model_indiv = sm.OLS(df['oil'], X_indiv).fit(cov_type='HAC', cov_kwds={'maxlags': 5})

print("Model A: AR(1) + Market + Individual Event Dummies")
print("=" * 70)
print(f"  alpha = {model_indiv.params['const']:.4f}, phi = {model_indiv.params['oil_lag']:.4f}, "
      f"beta = {model_indiv.params['market']:.4f}")
print(f"  R-squared = {model_indiv.rsquared:.4f}")
print(f"\n  Event Effects (avg daily AR in %, HAC s.e.):")
print(f"  {'Event':45s} {'gamma':>7s} {'t':>7s} {'CAR[-1,+1]':>10s}")
print(f"  {'-'*75}")
for i, row in opec_events.iterrows():
    col = f'D_{i}'
    g = model_indiv.params[col]
    t = model_indiv.tvalues[col]
    stars = '***' if abs(t) > 2.58 else ('**' if abs(t) > 1.96 else ('*' if abs(t) > 1.65 else ''))
    desc = row['decision'][:43]
    print(f"  {desc:45s} {g:+7.3f} {t:+7.3f}{stars:3s} {g*3:+10.3f}%")

# Model B: Pooled dummies
X_pooled = sm.add_constant(df[['oil_lag', 'market', 'D_bullish', 'D_bearish']])
model_pooled = sm.OLS(df['oil'], X_pooled).fit(cov_type='HAC', cov_kwds={'maxlags': 5})

print(f"\n\nModel B: AR(1) + Market + Pooled Event Dummies")
print("=" * 70)
print(model_pooled.summary2().tables[1].to_string())
g_bull = model_pooled.params['D_bullish']
g_bear = model_pooled.params['D_bearish']
print(f"\n  Interpretation:")
print(f"  Bullish OPEC decisions: avg daily AR = {g_bull:+.3f}%, CAR[-1,+1] = {g_bull*3:+.3f}%")
print(f"  Bearish OPEC decisions: avg daily AR = {g_bear:+.3f}%, CAR[-1,+1] = {g_bear*3:+.3f}%")

## 4. GARCH Models with Event Dummies

### 4.1 Why GARCH?

The AR model with event dummies assumes that the variance of $\varepsilon_t$ is constant over the entire sample. This assumption is unrealistic for financial return series, which exhibit volatility clustering: large returns (positive or negative) are followed by large returns, and small returns are followed by small returns. For oil returns, this clustering is especially pronounced around OPEC announcements, geopolitical shocks, and demand disruptions.

If volatility is time-varying and the model assumes it is constant, the OLS standard errors are incorrect. During high-volatility periods, the residuals are large but not abnormal (they reflect the elevated volatility, not an event effect). During low-volatility periods, a modest event effect may be economically significant but statistically insignificant because the constant-variance standard error is too large.

The GARCH(1,1) model of Bollerslev (1986) addresses this by modelling the conditional variance as a function of past squared residuals and past conditional variances:

$$
\varepsilon_t = \sigma_t z_t, \qquad z_t \sim N(0,1)
$$

$$
\sigma_t^2 = \omega + \alpha_1 \varepsilon_{t-1}^2 + \beta_1 \sigma_{t-1}^2 \tag{5}
$$

where $\omega > 0$, $\alpha_1 \geq 0$, $\beta_1 \geq 0$, and $\alpha_1 + \beta_1 < 1$ for stationarity. The conditional variance $\sigma_t^2$ is the market's one-step-ahead forecast of the return variance, given information up to time $t - 1$.

### 4.2 Event Dummies in the Mean and Variance Equations

The GARCH framework allows event dummies to enter both the conditional mean equation and the conditional variance equation:

**Mean equation:**

$$
R_{oil,t} = \alpha + \phi R_{oil,t-1} + \beta R_{m,t} + \sum_{k=1}^{K} \gamma_k D_{k,t} + \varepsilon_t \tag{6}
$$

**Variance equation:**

$$
\sigma_t^2 = \omega + \alpha_1 \varepsilon_{t-1}^2 + \beta_1 \sigma_{t-1}^2 + \sum_{k=1}^{K} \delta_k D_{k,t} \tag{7}
$$

The coefficient $\gamma_k$ measures the event's effect on the conditional mean (the abnormal return), exactly as in the OLS specification. The coefficient $\delta_k$ measures the event's effect on the conditional variance (event-induced volatility). If $\delta_k > 0$, the event increases volatility; if $\delta_k < 0$, the event reduces volatility (e.g., by resolving uncertainty).

This is a direct solution to the event-induced variance problem that motivated the BMP test in Session 4. Rather than using a robust variance estimator after the fact, the GARCH model estimates the variance change jointly with the mean effect, producing correctly sized test statistics by construction.

### 4.3 Estimation

GARCH models are estimated by maximum likelihood. The log-likelihood for the Gaussian GARCH(1,1) is:

$$
\ell(\theta) = -\frac{T}{2} \ln(2\pi) - \frac{1}{2} \sum_{t=1}^{T} \left[\ln(\sigma_t^2) + \frac{\varepsilon_t^2}{\sigma_t^2}\right]
$$

where $\theta = (\alpha, \phi, \beta, \gamma_1, \ldots, \gamma_K, \omega, \alpha_1, \beta_1, \delta_1, \ldots, \delta_K)$ is the full parameter vector. The maximization is performed numerically. We use the `arch` package in Python, which implements the GARCH family with flexible mean specifications and exogenous variables in both equations.

### 4.4 Interpretation

The t-statistics on $\gamma_k$ from the GARCH model are generally more reliable than those from OLS, because the GARCH likelihood correctly accounts for the time-varying volatility. In high-volatility periods, the GARCH model "expects" large residuals and assigns them less significance; in low-volatility periods, even modest abnormal returns are correctly identified as significant.

The variance-equation coefficients $\delta_k$ provide additional information that the standard event study cannot deliver: they measure the *uncertainty* effect of the event, not just the *price* effect. An OPEC meeting that resolves a long-running dispute may have a positive mean effect (prices rise because cuts are agreed) and a negative variance effect (uncertainty about future supply is reduced).

In [ ]:
# -- GARCH(1,1) with event dummies in mean and variance --

# Prepare data for arch package
# The arch package requires the exogenous variables as separate arrays

# Pooled specification for tractability (individual dummies in GARCH
# can cause convergence issues with many events)
exog_mean = df[['oil_lag', 'market', 'D_bullish', 'D_bearish']].values
exog_var = df[['D_bullish', 'D_bearish']].values

# Estimate GARCH(1,1) with exogenous variables in mean equation
# Using arch package: ARX(1)-GARCH(1,1)
from arch import arch_model

# Method: fit a GARCH with exogenous regressors in the mean
# arch_model expects the dependent variable and allows x= for mean regressors
am = arch_model(df['oil'].values, x=exog_mean, mean='ARX', lags=0,
                vol='GARCH', p=1, q=1, dist='normal')
garch_result = am.fit(disp='off', show_warning=False)

print("GARCH(1,1) with Event Dummies in Mean Equation")
print("=" * 65)
print(garch_result.summary().tables[0])
print()
print(garch_result.summary().tables[1])
print()
print(garch_result.summary().tables[2])

In [ ]:
# -- Comparison: OLS vs GARCH coefficients --

print("\\nComparison of Event Coefficients: OLS vs. GARCH")
print("=" * 65)
print(f"  {'Variable':15s}  {'OLS coef':>10s} {'OLS t':>8s}   {'GARCH coef':>10s} {'GARCH t':>8s}")
print(f"  {'-'*60}")

# Map GARCH parameter names (they depend on the arch package version)
garch_params = garch_result.params
garch_tvals = garch_result.tvalues

# Print the mean equation parameters
for var_name in ['const', 'D_bullish', 'D_bearish']:
    if var_name in model_pooled.params.index:
        ols_c = model_pooled.params[var_name]
        ols_t = model_pooled.tvalues[var_name]
    else:
        ols_c, ols_t = np.nan, np.nan

    # Find corresponding GARCH parameter
    garch_c, garch_t = np.nan, np.nan
    for gp_name in garch_params.index:
        if var_name.lower() in gp_name.lower() or var_name in gp_name:
            garch_c = garch_params[gp_name]
            garch_t = garch_tvals[gp_name]
            break

    print(f"  {var_name:15s}  {ols_c:+10.4f} {ols_t:+8.3f}   {garch_c:+10.4f} {garch_t:+8.3f}")

# Volatility persistence
if 'alpha[1]' in garch_params and 'beta[1]' in garch_params:
    persistence = garch_params['alpha[1]'] + garch_params['beta[1]']
    print(f"\\n  GARCH volatility persistence (alpha + beta): {persistence:.4f}")
    print(f"  Half-life of volatility shocks: {np.log(2) / (-np.log(persistence)):.1f} days")

In [ ]:
# -- Conditional volatility around OPEC events --

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

cond_vol = garch_result.conditional_volatility
dates = df.index

# Panel A: Oil returns with OPEC event markers
ax = axes[0]
ax.plot(dates, df['oil'], color='#1f78b4', linewidth=0.5, alpha=0.7)
for i, row in opec_events.iterrows():
    edate = row['date']
    if edate in dates or (dates.get_indexer([edate], method='ffill')[0] >= 0):
        color = '#d95f02' if row['direction'] > 0 else '#377eb8' if row['direction'] < 0 else 'gray'
        ax.axvline(edate, color=color, linewidth=1.5, alpha=0.5)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('Oil Return (%)')
ax.set_title('Panel A: Oil Returns with OPEC Announcements (orange = bullish, blue = bearish)')
ax.grid(True, alpha=0.2)

# Panel B: Conditional volatility
ax = axes[1]
ax.plot(dates, cond_vol, color='#e31a1c', linewidth=0.8)
for i, row in opec_events.iterrows():
    edate = row['date']
    color = '#d95f02' if row['direction'] > 0 else '#377eb8' if row['direction'] < 0 else 'gray'
    ax.axvline(edate, color=color, linewidth=1.5, alpha=0.5)
ax.set_ylabel('Conditional Volatility (%)')
ax.set_xlabel('Date')
ax.set_title('Panel B: GARCH(1,1) Conditional Volatility')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 5. Extensions

### 5.1 Asymmetric GARCH (GJR-GARCH)

Oil markets, like equity markets, exhibit a leverage effect: negative shocks tend to increase volatility more than positive shocks of the same magnitude. The GJR-GARCH model (Glosten, Jagannathan, and Runkle, 1993) captures this asymmetry by adding an indicator for negative shocks:

$$
\sigma_t^2 = \omega + (\alpha_1 + \kappa \mathbf{1}_{\varepsilon_{t-1} < 0}) \varepsilon_{t-1}^2 + \beta_1 \sigma_{t-1}^2
$$

where $\kappa > 0$ implies that negative shocks have a larger impact on volatility than positive shocks. In the oil market, negative demand shocks (recessions, COVID) tend to produce larger volatility spikes than positive supply shocks (OPEC cuts), so we expect $\kappa > 0$.

### 5.2 Student-t Innovations

Daily return distributions are heavy-tailed. Replacing the Gaussian assumption with a Student-$t$ distribution for $z_t$ in Equation (5) improves the fit and produces more reliable inference in the presence of outliers. The degrees-of-freedom parameter $\nu$ is estimated jointly with the other parameters. Values of $\nu$ between 4 and 8 are typical for daily financial returns, indicating substantially heavier tails than the normal distribution.

### 5.3 Multiple Event Windows

Rather than using a single $[-1, +1]$ dummy per event, one can use separate dummies for the pre-event day ($\tau = -1$), the event day ($\tau = 0$), and the post-event day ($\tau = +1$). This reveals the timing of the price adjustment: whether the market anticipates the announcement (pre-event drift), reacts immediately, or adjusts gradually.

In [ ]:
# -- GJR-GARCH with Student-t innovations --

am_gjr = arch_model(df['oil'].values, x=exog_mean, mean='ARX', lags=0,
                     vol='GARCH', p=1, o=1, q=1, dist='t')
gjr_result = am_gjr.fit(disp='off', show_warning=False)

print("GJR-GARCH(1,1,1) with Student-t Innovations")
print("=" * 65)
print(gjr_result.summary().tables[0])
print()
print(gjr_result.summary().tables[1])
print()
print(gjr_result.summary().tables[2])

# Extract key parameters
params = gjr_result.params
print(f"\nKey parameters:")
if 'gamma[1]' in params:
    print(f"  Asymmetry (gamma): {params['gamma[1]']:.4f} (t = {gjr_result.tvalues['gamma[1]']:.3f})")
    if params['gamma[1]'] > 0:
        print(f"  -> Negative shocks increase volatility more than positive shocks")
if 'nu' in params:
    print(f"  Student-t df (nu): {params['nu']:.2f}")
    print(f"  -> Heavy tails confirmed (normal would require nu -> infinity)")

## 6. Comparison with the Standard Event Study

To close the loop with Sessions 1 through 5, we now compare the regression-based results with the standard two-step event study applied to the same data. For each OPEC event, we estimate the market model over a 120-day estimation window (shorter than the usual 250 days, because OPEC events are frequent and we must avoid overlap), compute the CAR over $[-1, +1]$, and test significance using the standard cross-sectional t-test.

The comparison reveals two things: whether the event effect estimates are similar (they should be, by the Karafiath equivalence result), and whether the test statistics differ (they may, because the GARCH model produces better-calibrated standard errors).

In [ ]:
# -- Standard two-step event study for comparison --

est_len = 120
buf = 10
ew = (-1, 1)
ed = list(range(ew[0], ew[1] + 1))

standard_cars = []
for i, row in opec_events.iterrows():
    edate = row['date']
    eidx = trading_days.get_indexer([edate], method='ffill')[0]
    ev_s = eidx + ew[0]
    ev_e = eidx + ew[1]
    est_e = ev_s - buf - 1
    est_s = est_e - est_len + 1

    if est_s < 0 or ev_e >= len(trading_days):
        continue

    y_est = df['oil'].iloc[est_s:est_e+1].values
    x_est = df['market'].iloc[est_s:est_e+1].values
    valid = ~(np.isnan(y_est) | np.isnan(x_est))
    if valid.sum() < 30:
        continue
    X = sm.add_constant(x_est[valid])
    ols = sm.OLS(y_est[valid], X).fit()
    a, b = ols.params
    sig = np.sqrt(ols.mse_resid)

    y_ev = df['oil'].iloc[ev_s:ev_e+1].values
    x_ev = df['market'].iloc[ev_s:ev_e+1].values
    ar = y_ev - (a + b * x_ev)
    car = np.nansum(ar)

    standard_cars.append({
        'date': edate,
        'decision': row['decision'][:40],
        'direction': row['direction'],
        'CAR_standard': car,
        'sigma': sig,
    })

scar_df = pd.DataFrame(standard_cars)

# Merge with regression-based estimates
print("Standard vs. Regression-Based Event Study")
print("=" * 80)
print(f"  {'Decision':42s} {'Standard':>10s} {'Regress.':>10s} {'Direction':>10s}")
print(f"  {'-'*75}")

for i, row in opec_events.iterrows():
    col = f'D_{i}'
    if col in model_indiv.params:
        reg_car = model_indiv.params[col] * 3  # 3-day window
    else:
        reg_car = np.nan

    match = scar_df[scar_df['date'] == row['date']]
    if len(match) > 0:
        std_car = match.iloc[0]['CAR_standard']
    else:
        std_car = np.nan

    desc = row['decision'][:40]
    dir_label = 'Bullish' if row['direction'] > 0 else ('Bearish' if row['direction'] < 0 else 'Neutral')
    print(f"  {desc:42s} {std_car:+10.3f}% {reg_car:+10.3f}% {dir_label:>10s}")

## 7. Summary and Preview of Session 9

This session developed the regression-based event study, where event dummies are embedded directly in a time-series model of returns. The key results are:

The regression approach with event dummies (Karafiath, 1988) is algebraically equivalent to the standard two-step event study when applied to a single event, but it extends naturally to multiple events for a single entity, producing joint estimates of all event effects from a single model.

The GARCH framework (Bollerslev, 1986) addresses event-induced variance by modelling the conditional variance as time-varying. Event dummies can enter both the mean equation (measuring the abnormal return) and the variance equation (measuring the volatility change). The t-statistics from the GARCH model are correctly calibrated to the local volatility environment, producing more reliable inference than OLS in the presence of volatility clustering.

The GJR-GARCH extension captures the asymmetry between positive and negative shocks, and Student-t innovations accommodate the heavy tails of daily returns. Both extensions improve the model's fit and the reliability of the event effect estimates.

The application to OPEC production announcements and oil returns illustrated all these features: multiple events for a single asset, large mean effects (production cuts raise oil prices), variance effects (events resolve or create uncertainty), and the importance of conditioning on the volatility regime when assessing statistical significance.

Session 9 brings together the full toolkit in a practical guide to designing, implementing, and reporting an event study from start to finish.

**Additional references:**

- Glosten, L.R., Jagannathan, R. and Runkle, D.E. (1993). On the Relation between the Expected Value and the Volatility of the Nominal Excess Return on Stocks. *Journal of Finance*, 48(5), 1779--1801.
- Hansen, P.R. and Lunde, A. (2005). A Forecast Comparison of Volatility Models: Does Anything Beat a GARCH(1,1)? *Journal of Applied Econometrics*, 20(7), 873--889.
- Karafiath, I. (1988). Using Dummy Variables in the Event Methodology. *Financial Review*, 23(3), 351--357.
- Pynnonen, S. (2005). On Regression Based Event Study. *Acta Wasaensia*, 143, 327--354.